Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.
SPDX-License-Identifier: MIT-0

# Platform signals

Modules 2, 3a and 3b each left a trail: LiteLLM request logs on ECS, the interceptor Lambdas' audit
records, the AgentCore Gateway's own spans. The Dataset integration forwards all of them into one table,
`default`, that you query with SQL. Omni backfilled up to seven days of history, so the Module 3b gateway
tests are already there.

**Tracks 2 and 3.** Track 1 continues with `05-agent-traces.ipynb`.

In [ ]:
%pip install --quiet "boto3==1.43.126" "botocore==1.43.126"
import boto3

assert tuple(int(x) for x in boto3.__version__.split(".")[:3]) >= (1, 43, 100), (
    f"boto3 too old: {boto3.__version__} (need >=1.43.100 for the cloudwatchomni client)"
)
# If the kernel imported an older botocore before the pip install above, the new
# service models are not loaded yet. Fail here with an actionable message.
services = boto3.session.Session().get_available_services()
assert "cloudwatchomni" in services and "observabilityadmin" in services, (
    "This kernel has no cloudwatchomni client. Restart the kernel (Kernel > Restart) and run this cell again."
)
print(f"boto3 {boto3.__version__} OK - cloudwatchomni and observabilityadmin clients available")

In [ ]:
import json
import os
import pathlib
import time

import boto3

REGION = (
    os.environ.get("AWS_REGION")
    or os.environ.get("AWS_DEFAULT_REGION")
    or boto3.session.Session().region_name
)
if not REGION:
    raise RuntimeError("No AWS region configured. Set AWS_REGION or run `aws configure set region <region>`.")
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

# The CLI pages save identifiers as `export KEY=VALUE` lines in this file, so a
# participant can switch between the CLI and notebook paths at any step.
STATE_FILE = pathlib.Path.home() / ".workshop-omni.env"


def load_state():
    state = {}
    if STATE_FILE.exists():
        for line in STATE_FILE.read_text().splitlines():
            if line.startswith("export ") and "=" in line:
                key, value = line[len("export "):].split("=", 1)
                state[key] = value.strip()
    return state


def save_state(**values):
    state = load_state()
    state.update({k: v for k, v in values.items() if v})
    STATE_FILE.write_text("".join(f"export {k}={v}\n" for k, v in state.items()))
    return state


omni = boto3.client("cloudwatchomni", region_name=REGION)
obs_admin = boto3.client("observabilityadmin", region_name=REGION)
logs = boto3.client("logs", region_name=REGION)
cfn = boto3.client("cloudformation", region_name=REGION)
print(f"Region {REGION}, account ...{ACCOUNT_ID[-4:]}")

## Step 1: A query session and a helper

Telemetry queries run inside a session whose name is unique per account. Every query needs an explicit
time range; the Dataset scans the whole range, so keep it tight.

In [ ]:
from datetime import datetime, timedelta, timezone

QUERY_SESSION = omni.start_telemetry_query_session(sessionName=f"workshop-nb-{int(time.time())}")["sessionId"]
now = datetime.now(timezone.utc)
START = (now - timedelta(hours=6)).strftime("%Y-%m-%dT%H:%M:%SZ")
END = (now + timedelta(minutes=5)).strftime("%Y-%m-%dT%H:%M:%SZ")
TIME_RANGE = f"`@timestamp` BETWEEN to_timestamp('{START}') AND to_timestamp('{END}')"


def run_query(sql, max_wait_s=60):
    qid = omni.start_telemetry_query(queryString=sql, sessionId=QUERY_SESSION)["queryId"]
    deadline = time.time() + max_wait_s
    while True:
        result = omni.get_telemetry_query_results(queryId=qid)
        if result["status"] != "Running" or time.time() > deadline:
            break
        time.sleep(2)
    rows = result.get("rows", [])
    print(f"status={result['status']}  rows={len(rows)}  scanned={result.get('statistics', {}).get('bytesScanned')} bytes")
    for row in rows:
        print("  " + "  ".join(f"{k}={v}" for k, v in row.items()))
    return rows


print(f"query session {QUERY_SESSION}, range {START} .. {END}")

## Step 2: Records per log group (one line per platform layer)

In [ ]:
rows = run_query(f"""
SELECT `@logGroupName` AS log_group, COUNT(*) AS records
FROM default
WHERE {TIME_RANGE}
GROUP BY 1 ORDER BY records DESC LIMIT 25
""")
if not any(r.get("log_group") == "aws/spans" for r in rows):
    print("\nNo aws/spans rows yet: run the Module 3b step 6 gateway test once and re-run this cell.")

## Step 3: Follow one tool call across layers

The request interceptor logs the caller identity and tool name for every `tools/call`. The same identity
appears on the Gateway spans in `aws/spans`, which is what lets Omni join the two in the UI.

In [ ]:
run_query(f"""
SELECT `@timestamp` AS at, `@logGroupName` AS log_group, `@message` AS message
FROM default
WHERE {TIME_RANGE}
  AND `@logGroupName` LIKE '/aws/lambda/%gateway-request-interceptor'
  AND `@message` LIKE '%tools/call%'
ORDER BY `@timestamp` DESC LIMIT 5
""")

## Step 4: Ask the question in plain language

In your space, open **Explore**, choose **Ask Omni**, and type: *Which gateway tools were called most in
the last six hours, and by which client?* The Omni agent writes the SQL, runs it over the same `default`
table and shows the query next to the answer. Then open **Applications** for the service map Omni derives
from the telemetry.

| Layer | Records come from | What a platform team reads there |
|-------|-------------------|----------------------------------|
| LLM Gateway (Module 2) | ECS task logs | Which virtual key called which model, guardrail blocks |
| Interceptors (Modules 3a, 3b) | Lambda logs | Caller identity per tool call, access-control decisions |
| AgentCore Gateway (Module 3b) | Spans in `aws/spans` | Latency and errors per tool, joined to the audit trail |
| Tool Lambdas | Lambda logs and spans | Downstream failures behind a slow tool |

Track 2 continues with `08-cleanup.ipynb`; Track 3 with `05-agent-traces.ipynb`.

In [ ]:
omni.stop_telemetry_query_session(sessionId=QUERY_SESSION)
print("query session closed")